# Fase 5 Datathon Passos Mágicos: limpeza e preparação do painel PEDE

Este notebook corresponde à primeira etapa da Fase 5 (Datathon *"Case Passos Mágicos"*).

O objetivo desta etapa é preparar e validar os dados que serão utilizados nas análises exploratórias e na modelagem preditiva. Partimos do arquivo bruto `data/raw/base_bronze.xlsx`, que contém três abas, `PEDE2022`, `PEDE2023` e `PEDE2024`, referentes às diferentes edições da Pesquisa Extensiva do Desenvolvimento Educacional (PEDE).

Como a estrutura das bases apresenta diferenças entre os anos, primeiro identificamos essas diferenças e, em seguida, harmonizamos os dados em um único painel longitudinal. Nesse painel, cada registro representa um aluno em um determinado ano, identificado de forma anônima e estável pela coluna `RA`.

Ao final deste notebook, teremos um painel consolidado, padronizado e validado, salvo em `data/processed/pede_painel_consolidado.csv`.

A lógica de transformação e harmonização está centralizada em `src/data_prep.py`. Este notebook é responsável por executar essas funções, realizar verificações de qualidade e documentar as principais decisões tomadas durante a preparação dos dados.

In [ ]:
# Importa bibliotecas e funções do projeto para carregar, padronizar,
# inspecionar e preparar os dados, além de configurar o estado aleatório
# e a visualização das colunas do DataFrame.

import sys
import pandas as pd

sys.path.append("..")
from src.data_prep import (
    RANDOM_STATE,
    build_painel,
    build_painel_com_alvo,
    calcular_pedra_por_inde,
    carregar_abas_brutas,
    inspecionar_abas,
    padronizar_nomes_e_categorias,
)

pd.set_option("display.max_columns", 60)
RANDOM_STATE

42

## 1. Inspeção das 3 abas

### Objetivo desta etapa

Antes de realizar qualquer transformação, precisamos entender como os dados estão estruturados em cada ano.

Nesta etapa, inspecionamos as abas `PEDE2022`, `PEDE2023` e `PEDE2024` para identificar:

- quantidade de registros e colunas;
- nomes das variáveis disponíveis em cada ano;
- diferenças de nomenclatura para indicadores equivalentes;
- mudanças no formato de determinadas variáveis;
- indicadores que passaram a existir ou deixaram de ser preenchidos ao longo do período.

Essa inspeção é importante porque não podemos assumir que as três bases possuem exatamente a mesma estrutura. As diferenças identificadas aqui orientam as regras utilizadas posteriormente na harmonização dos dados.

In [ ]:
# Carrega as abas da base Bronze e realiza uma inspeção detalhada,
# gerando um resumo das características de cada aba.

abas_brutas = carregar_abas_brutas("../data/raw/base_bronze.xlsx")
resumo_abas = inspecionar_abas(abas_brutas, verbose=True)

PEDE2022: 860 linhas x 42 colunas
['RA', 'Fase', 'Turma', 'Nome', 'Ano nasc', 'Idade 22', 'Gênero', 'Ano ingresso', 'Instituição de ensino', 'Pedra 20', 'Pedra 21', 'Pedra 22', 'INDE 22', 'Cg', 'Cf', 'Ct', 'Nº Av', 'Avaliador1', 'Rec Av1', 'Avaliador2', 'Rec Av2', 'Avaliador3', 'Rec Av3', 'Avaliador4', 'Rec Av4', 'IAA', 'IEG', 'IPS', 'Rec Psicologia', 'IDA', 'Matem', 'Portug', 'Inglês', 'Indicado', 'Atingiu PV', 'IPV', 'IAN', 'Fase ideal', 'Defas', 'Destaque IEG', 'Destaque IDA', 'Destaque IPV']
PEDE2023: 1014 linhas x 48 colunas
['RA', 'Fase', 'INDE 2023', 'Pedra 2023', 'Turma', 'Nome Anonimizado', 'Data de Nasc', 'Idade', 'Gênero', 'Ano ingresso', 'Instituição de ensino', 'Pedra 20', 'Pedra 21', 'Pedra 22', 'Pedra 23', 'INDE 22', 'INDE 23', 'Cg', 'Cf', 'Ct', 'Nº Av', 'Avaliador1', 'Rec Av1', 'Avaliador2', 'Rec Av2', 'Avaliador3', 'Rec Av3', 'Avaliador4', 'Rec Av4', 'IAA', 'IEG', 'IPS', 'IPP', 'Rec Psicologia', 'IDA', 'Mat', 'Por', 'Ing', 'Indicado', 'Atingiu PV', 'IPV', 'IAN', 'Fase 

### O que a inspeção acima mostra

A inspeção inicial evidencia diferenças importantes entre as três edições da base:

- **Estrutura das colunas:** as três abas possuem quantidades diferentes de colunas (42, 48 e 50) e alguns indicadores mudam de nome entre os anos, mesmo quando representam o mesmo conceito. Por exemplo, o INDE do próprio ano aparece como `INDE 22` em `PEDE2022`, `INDE 2023` em `PEDE2023` e `INDE 2024` em `PEDE2024`.

- **Indicador IPP:** aparece nas bases de 2023 e 2024, mas não em 2022. Portanto, sua ausência em 2022 deve ser tratada como uma diferença estrutural da fonte, e não como uma falha de leitura ou transformação.

- **Variável Fase:** apresenta formatos diferentes entre os anos. Em `PEDE2022`, `Fase` aparece como valor numérico de 0 a 7. Em `PEDE2023`, aparecem valores como `"ALFA"` e `"FASE N"`. Em `PEDE2024`, a informação pode aparecer combinada com a identificação da turma, como `"1A"` ou `"2B"`, ou como um número isolado, como `"9"`. Por isso, a extração do nível numérico da fase precisa considerar explicitamente os formatos encontrados em cada ano.

- **Variável Fase ideal:** também apresenta diferenças de nomenclatura, aparecendo como `Fase ideal` em 2022 e `Fase Ideal` em 2023 e 2024.

Essas diferenças justificam a necessidade de uma etapa específica de harmonização antes que os dados possam ser analisados conjuntamente.

## 2. Harmonização: `build_painel()`

### Objetivo desta etapa

As três bases possuem diferenças de nomenclatura e formato, mas precisamos analisá-las como um único painel longitudinal.

A função `build_painel()`, implementada em `src/data_prep.py`, realiza essa harmonização. O objetivo é transformar as três abas em uma estrutura comum, preservando as informações disponíveis na fonte e mantendo uma linha por aluno em cada ano.

Para cada ano, são realizadas as seguintes etapas:

1. Renomeação das colunas relevantes para um esquema comum em `snake_case`, como `ra`, `fase_num`, `inde`, `pedra`, `ian`, `ida`, `ieg`, `iaa`, `ips`, `ipp`, `ipv`, notas e dados demográficos.

2. Extração do nível numérico de `fase_num` e `fase_ideal_num`, considerando o formato específico encontrado em cada edição da base. Essa transformação é realizada pela função `_extrair_numero_fase`.

3. Cálculo da `defasagem_calculada`, utilizando a diferença entre a fase do aluno e a fase esperada:

   `defasagem_calculada = fase_num - fase_ideal_num`

4. Empilhamento das três bases em uma única estrutura longitudinal, com a coluna `ano` identificando se o registro pertence a 2022, 2023 ou 2024.

O resultado dessa etapa é uma base padronizada que permite comparar os mesmos indicadores ao longo dos anos.

In [ ]:
# Constrói o painel harmonizado a partir da base Bronze e compara seu volume
# com as abas brutas, exibindo dimensões, quantidade de alunos, anos disponíveis
# e uma amostra dos primeiros registros.

painel = build_painel("../data/raw/base_bronze.xlsx", verbose=False)

n_linhas_brutas = sum(len(df) for df in abas_brutas.values())
print(f"Linhas nas 3 abas brutas somadas: {n_linhas_brutas}")
print(f"Linhas no painel harmonizado:      {len(painel)}")
print(f"Shape do painel: {painel.shape}")
print()
print("Alunos (RA) distintos:", painel["ra"].nunique())
print("Anos presentes:", sorted(painel["ano"].unique().tolist()))
print()
painel.head()

Linhas nas 3 abas brutas somadas: 3030
Linhas no painel harmonizado:      3030
Shape do painel: (3030, 26)

Alunos (RA) distintos: 1661
Anos presentes: [2022, 2023, 2024]



,ra,ano,fase_num,fase_ideal_num,defasagem_fornecida,inde,pedra,ian,ida,ieg,iaa,ips,ipp,ipv,nota_matematica,nota_portugues,nota_ingles,indicado,atingiu_pv,turma,genero,ano_ingresso,instituicao_ensino,ano_nascimento,defasagem_calculada,idade_anos
0,1,2022,7.0,8.0,-1,5.783,Quartzo,5.0,4.0,4.1,8.3,5.6,NaN,7.278,2.7,3.5,6.0,Sim,Não,A,Feminino,2016,Escola Pública,2003,-1.0,19
1,1,2023,8.0,8.0,0,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8E,Feminino,2016,Privada *Parcerias com Bolsa 100%,2003,0.0,20
2,1,2024,8.0,8.0,0,NaN,NaN,10.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8E,Feminino,2021,Privada *Parcerias com Bolsa 100%,2003,0.0,21
3,2,2022,7.0,7.0,0,7.055,Ametista,10.0,6.8,5.2,8.8,6.3,NaN,6.778,6.3,4.5,9.7,Não,Não,A,Feminino,2017,Rede Decisão,2005,0.0,17
4,2,2023,8.0,8.0,0,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8E,Feminino,2017,Privada *Parcerias com Bolsa 100%,2005,0.0,18


In [ ]:
# Exibe os tipos de dados de cada coluna do painel para verificar sua estrutura.

print("Tipos de dado por coluna:")
painel.dtypes

Tipos de dado por coluna:


ra                       int64
ano                      int64
fase_num               float64
fase_ideal_num         float64
defasagem_fornecida      int64
inde                   float64
pedra                   object
ian                    float64
ida                    float64
ieg                    float64
iaa                    float64
ips                    float64
ipp                    float64
ipv                    float64
nota_matematica        float64
nota_portugues         float64
nota_ingles            float64
indicado                object
atingiu_pv              object
turma                   object
genero                  object
ano_ingresso             int64
instituicao_ensino      object
ano_nascimento           int64
defasagem_calculada    float64
idade_anos               int64
dtype: object

## 3. QA: `defasagem_calculada` vs. defasagem fornecida pela Passos Mágicos

### Objetivo desta validação

Depois de harmonizar as diferentes formas de registro da variável `Fase`, precisamos verificar se a transformação realizada está produzindo um resultado consistente.

A Passos Mágicos já fornece uma medida de defasagem na base original: `Defas` em 2022 e `Defasagem` em 2023/2024. Durante a harmonização, essas colunas são padronizadas como `defasagem_fornecida`.

Também calculamos uma segunda medida, chamada `defasagem_calculada`, a partir de:

`fase_num - fase_ideal_num`

A comparação entre as duas medidas funciona como um controle de qualidade da transformação realizada. Se a extração das fases estiver correta, esperamos que a defasagem calculada seja, em geral, consistente com a informação originalmente fornecida pela fonte.

In [ ]:
# Compara a defasagem calculada com a fornecida, medindo a taxa de concordância
# e a correlação entre elas, e identifica as linhas onde os valores divergem.

comparavel = painel.dropna(subset=["defasagem_calculada", "defasagem_fornecida"])
taxa_match = (comparavel["defasagem_calculada"] == comparavel["defasagem_fornecida"]).mean()
correlacao = comparavel["defasagem_calculada"].corr(comparavel["defasagem_fornecida"])

print(f"Linhas comparáveis (ambas as colunas preenchidas): {len(comparavel)} de {len(painel)}")
print(f"% de linhas com defasagem_calculada == defasagem_fornecida: {taxa_match:.2%}")
print(f"Correlação (Pearson) entre as duas: {correlacao:.4f}")

divergentes = comparavel[comparavel["defasagem_calculada"] != comparavel["defasagem_fornecida"]]
print(f"\nLinhas divergentes: {len(divergentes)}")
divergentes[["ra", "ano", "fase_num", "fase_ideal_num", "defasagem_calculada", "defasagem_fornecida"]]

Linhas comparáveis (ambas as colunas preenchidas): 3030 de 3030
% de linhas com defasagem_calculada == defasagem_fornecida: 99.93%
Correlação (Pearson) entre as duas: 0.9960

Linhas divergentes: 2


,ra,ano,fase_num,fase_ideal_num,defasagem_calculada,defasagem_fornecida
2884,1516,2024,3.0,3.0,0.0,3
2887,1519,2024,3.0,3.0,0.0,3


### Leitura do resultado

A concordância entre `defasagem_calculada` e `defasagem_fornecida` é muito alta, indicando que a extração e a transformação das diferentes representações de `Fase` estão, em geral, consistentes com a informação fornecida pela fonte.

As poucas divergências identificadas representam uma fração mínima da base. Nesses casos, observamos `fase_num == fase_ideal_num`, resultando em `defasagem_calculada = 0`, enquanto `defasagem_fornecida` apresenta outro valor.

O padrão observado não indica um erro sistemático de parsing concentrado em determinado ano ou formato de Fase. No entanto, a origem dessas divergências não pode ser determinada apenas por esta validação. Por isso, elas são mantidas como uma diferença entre a variável calculada e a variável fornecida pela fonte, sem assumir uma causa específica.

Para as etapas seguintes, `defasagem_calculada` será utilizada quando for necessário aplicar uma regra explícita baseada em `fase_num` e `fase_ideal_num`.

## 4. QA: Pedra recalculada a partir do INDE vs. Pedra fornecida

### Objetivo desta validação

A variável `Pedra` já está presente nas bases originais. Como também temos o `INDE`, podemos verificar se a classificação fornecida é reproduzível a partir das faixas de INDE documentadas utilizadas nesta análise.

Para isso, recalculamos a Pedra utilizando `calcular_pedra_por_inde`, baseada nas faixas definidas em `FAIXAS_PEDRA_POR_INDE`:

- Quartzo: 2,405–5,506
- Ágata: 5,506–6,868
- Ametista: 6,868–8,230
- Topázio: 8,230–9,294

Em seguida, comparamos a classificação recalculada com a `Pedra` originalmente fornecida pela Passos Mágicos.

Esta etapa não tem como objetivo substituir a classificação original. Ela funciona como uma validação para verificar o quanto a regra documentada reproduz a classificação presente na base.

In [ ]:
# Recalcula a Pedra a partir do INDE, compara com a Pedra fornecida
# e apresenta a taxa de concordância e a matriz de divergências entre os valores.

comparavel_pedra = painel.dropna(subset=["inde", "pedra"]).copy()
comparavel_pedra["pedra_recalculada"] = calcular_pedra_por_inde(comparavel_pedra["inde"])

taxa_match_pedra = (comparavel_pedra["pedra"] == comparavel_pedra["pedra_recalculada"]).mean()
print(f"Linhas comparáveis (INDE e Pedra preenchidos): {len(comparavel_pedra)} de {len(painel)}")
print(f"% de linhas com Pedra recalculada == Pedra fornecida: {taxa_match_pedra:.2%}")
print()
print("Crosstab Pedra fornecida (linha) x Pedra recalculada (coluna):")
pd.crosstab(comparavel_pedra["pedra"], comparavel_pedra["pedra_recalculada"])

Linhas comparáveis (INDE e Pedra preenchidos): 2845 de 3030
% de linhas com Pedra recalculada == Pedra fornecida: 81.51%

Crosstab Pedra fornecida (linha) x Pedra recalculada (coluna):


pedra_recalculada,Ametista,Quartzo,Topázio,Ágata
pedra,,,,
Ametista,1120,0,0,0
Quartzo,0,149,0,167
Topázio,219,0,457,0
Ágata,128,0,0,593


### Leitura do resultado

O nível de concordância é consideravelmente menor do que o observado na validação da defasagem, ficando na faixa de 80%.

O crosstab mostra que as divergências não estão distribuídas de forma aleatória. Elas se concentram principalmente nas fronteiras entre categorias consecutivas: parte dos registros classificados como `Quartzo` na fonte é classificada como `Ágata` pela regra recalculada, enquanto parte dos `Topázio` é recalculada como `Ametista`.

Esse padrão indica que as faixas utilizadas para o recálculo não reproduzem integralmente a regra que gerou a classificação fornecida na base.

Uma possível explicação é a existência de diferenças de corte, arredondamento ou versão da regra utilizada em diferentes ciclos. Entretanto, essa hipótese não pode ser confirmada apenas pelos dados disponíveis neste notebook.

Por esse motivo, a `Pedra` fornecida pela fonte é preservada como informação original. A classificação recalculada deve ser utilizada apenas como instrumento de QA e não como substituta da variável original.

## 5. Mapeamento de dados ausentes por coluna e por ano

### Objetivo desta etapa

Nem toda ausência de informação representa um problema de qualidade dos dados.

Alguns indicadores foram introduzidos ou deixaram de ser preenchidos ao longo dos anos, gerando **ausência estrutural**. Outros possuem valores faltantes mesmo nos anos em que estavam disponíveis, caracterizando uma ausência de preenchimento ou avaliação.

Por isso, analisamos a porcentagem de valores ausentes de cada coluna separadamente para 2022, 2023 e 2024.

Essa distinção é importante para evitar que uma variável estruturalmente ausente em determinado ano seja interpretada como um problema de coleta ou, posteriormente, seja tratada como se representasse um valor real.

In [ ]:
# Calcula o percentual de valores ausentes em cada coluna, separando os resultados por ano,
# e apresenta a tabela arredondada para três casas decimais.

percentual_ausente_por_ano = pd.concat(
    {
        ano: painel.loc[painel["ano"] == ano].drop(columns=["ra", "ano"]).isna().mean()
        for ano in sorted(painel["ano"].unique())
    },
    axis=1,
).round(3)
percentual_ausente_por_ano

,2022,2023,2024
fase_num,0.000,0.000,0.000
fase_ideal_num,0.000,0.000,0.000
defasagem_fornecida,0.000,0.000,0.000
inde,0.000,0.082,0.088
pedra,0.000,0.082,0.088
ian,0.000,0.000,0.000
ida,0.000,0.076,0.087
ieg,0.000,0.075,0.000
iaa,0.000,0.062,0.088
ips,0.000,0.068,0.088


### Leitura do mapa de ausência

O mapa de ausência permite identificar diferentes padrões entre os indicadores:

- `ipp`: apresenta **100% de ausência em 2022** e passa a estar disponível em 2023/2024. Esse comportamento caracteriza uma ausência estrutural, relacionada à disponibilidade do indicador na fonte.

- `indicado` e `atingiu_pv`: apresentam preenchimento em 2022, mas estão **100% ausentes em 2023 e 2024**. Portanto, esses indicadores foram registrados na edição de 2022, mas não estão disponíveis nas edições posteriores.

- `nota_ingles`: apresenta uma taxa elevada de ausência nos três anos. Como o padrão ocorre de forma recorrente ao longo do período, não se trata de uma ausência específica de determinada edição. O dado deve, portanto, permanecer como ausente na base, sem assumir que a ausência representa nota zero.

- Os indicadores centrais (`inde`, `ida`, `ieg`, `iaa`, `ips`, `ipp`, `ipv` e notas) apresentam ausência baixa, embora não nula, em 2023/2024 e praticamente nenhuma ausência em 2022.

Esses padrões serão considerados nas etapas seguintes, principalmente na análise exploratória e na definição das variáveis utilizadas na modelagem.

In [ ]:
# Analisa as linhas com INDE ausente que possuem IPP preenchido,
# verificando a presença ou ausência dos demais componentes do INDE.

componentes_inde = ["ian", "ida", "ieg", "iaa", "ips", "ipp", "ipv"]

inde_ausente = painel[painel["inde"].isna()]
com_ipp_presente = inde_ausente[inde_ausente["ipp"].notna()]

print(f"Linhas com INDE ausente: {len(inde_ausente)}")
print(f"Dessas, com IPP presente: {len(com_ipp_presente)}")
print()
print("Ausência (True) dos demais componentes do INDE nas linhas com INDE ausente e IPP presente:")
com_ipp_presente[componentes_inde].isna()

Linhas com INDE ausente: 185
Dessas, com IPP presente: 7

Ausência (True) dos demais componentes do INDE nas linhas com INDE ausente e IPP presente:


,ian,ida,ieg,iaa,ips,ipp,ipv
300,False,False,False,False,True,False,False
534,False,False,False,False,True,False,False
551,False,False,False,False,True,False,False
554,False,False,False,False,True,False,False
2390,False,True,False,False,False,False,False
2492,False,False,False,False,True,False,False
2494,False,False,False,False,True,False,False


### O `ipp` pode ser utilizado para recompor o `INDE` ausente?

Como o `INDE` é um indicador composto, avaliamos se a disponibilidade do `ipp` permitiria recuperar registros em que o `INDE` está ausente.

A análise mostra que:

- `ipp` está **100% ausente em 2022**, pois o indicador não estava disponível nessa edição;
- em 2023/2024, o `ipp` também apresenta ausência em parte dos registros;
- entre as 185 linhas com `inde` ausente, apenas 7 possuem `ipp` preenchido;
- nessas 7 linhas, outro componente necessário para a composição do INDE também está ausente, especificamente `ida` ou `ips`.

Portanto, a disponibilidade do `ipp` não é suficiente para recompor os registros com `inde` ausente.

Também foi considerada a possibilidade de estimar o `ipp` de 2022 algebricamente a partir do `INDE` e dos demais componentes. Essa abordagem foi descartada porque a fórmula de sete componentes foi validada com dados de 2023/2024 e não foi estabelecido neste notebook que a mesma regra possa ser aplicada retroativamente a 2022.

Além disso, mesmo que essa reconstrução fosse possível, o valor estimado seria apenas uma combinação das informações já presentes no painel e não acrescentaria uma nova informação observada.

A decisão sobre como tratar a disponibilidade do `ipp` na modelagem, por exemplo, exclusão da variável, imputação ou criação de um indicador de disponibilidade, será tomada no notebook de modelagem.

## 6. Alvo: `build_painel_com_alvo()`

### Objetivo desta etapa

A modelagem da Fase 5 tem como objetivo identificar o risco de defasagem futura. Por isso, o alvo do modelo não pode representar a situação do aluno no mesmo ano em que as características são observadas.

A função `build_painel_com_alvo()` cria a variável `alvo_risco_defasagem_prox_ano` utilizando a situação do aluno no ano seguinte:

- `1`: o aluno está defasado no ano seguinte (`defasagem_calculada < 0`);
- `0`: o aluno não está defasado no ano seguinte;
- `NA`: não existe registro do aluno no ano seguinte.

O relacionamento é realizado por `RA` e pelo ano seguinte (`ano + 1`). Dessa forma, as características disponíveis em um determinado ano são associadas ao resultado observado posteriormente.

### Por que utilizar o ano seguinte?

A variável `defasagem_calculada` descreve a situação atual do aluno:

`fase_num - fase_ideal_num`

Se o resultado for negativo, o aluno está abaixo da fase esperada. Se for zero ou positivo, está na fase esperada ou acima dela.

Entretanto, para um modelo preditivo, a pergunta é diferente:

> Dadas as informações disponíveis sobre o aluno hoje, ele estará defasado no ano seguinte?

Essa definição permite separar claramente as informações utilizadas como entrada do modelo do resultado que queremos prever.

### Por que não utilizar a defasagem do mesmo ano como alvo?

Se utilizássemos a defasagem do próprio ano como variável-alvo, estaríamos descrevendo uma situação que já conhecemos, e não fazendo uma previsão.

Além disso, algumas variáveis do mesmo período poderiam fornecer diretamente ou indiretamente a informação utilizada para construir o alvo, criando risco de vazamento de dados (*data leakage*).

O valor do modelo preditivo está justamente em utilizar informações disponíveis no período atual para estimar uma situação futura, permitindo que a identificação do risco possa ocorrer antes do resultado observado.

### Exemplo

Considere o seguinte histórico:

| Ano | defasagem_calculada | alvo_risco_defasagem_prox_ano |
|---|---:|---:|
| 2022 | -1 (defasado) | 0 |
| 2023 | 0 (em dia) | 0 |
| 2024 | 0 (em dia) | NaN |

Na linha de 2022, o aluno estava defasado naquele momento (`-1`). Entretanto, o alvo dessa linha é `0`, porque o alvo não representa a situação de 2022: ele representa o que aconteceu em 2023.

Como a defasagem observada em 2023 foi `0`, o aluno não estava defasado no ano seguinte e, portanto, o alvo de 2022 é `0`.

Da mesma forma, a linha de 2024 possui `NaN` porque não existe `PEDE2025` nesta base. Sem o registro do ano seguinte, não é possível determinar qual foi o resultado futuro do aluno.

In [ ]:
# Cria o painel com a variável-alvo de risco de defasagem no próximo ano,
# verifica se o número de linhas foi preservado e analisa, por ano, a
# quantidade e a distribuição dos alunos com alvo definido.

painel_com_alvo = build_painel_com_alvo(painel)

print(f"Shape do painel com alvo: {painel_com_alvo.shape}")
print(f"(painel original tinha {painel.shape[0]} linhas, build_painel_com_alvo não deve alterar o nº de linhas)\n")

for ano in sorted(painel_com_alvo["ano"].unique()):
    fatia_ano = painel_com_alvo.loc[painel_com_alvo["ano"] == ano, "alvo_risco_defasagem_prox_ano"]
    n_total = len(fatia_ano)
    n_definido = fatia_ano.notna().sum()
    print(f"ano={ano}: {n_definido}/{n_total} alunos ({n_definido / n_total:.1%}) com alvo definido")
    if n_definido > 0:
        distrib = fatia_ano.value_counts(normalize=True).sort_index()
        print(f"    dentre os definidos -> {distrib.to_dict()}")

Shape do painel com alvo: (3030, 27)
(painel original tinha 3030 linhas, build_painel_com_alvo não deve alterar o nº de linhas)

ano=2022: 600/860 alunos (69.8%) com alvo definido
    dentre os definidos -> {0.0: 0.39, 1.0: 0.61}
ano=2023: 765/1014 alunos (75.4%) com alvo definido
    dentre os definidos -> {0.0: 0.5973856209150327, 1.0: 0.40261437908496733}
ano=2024: 0/1156 alunos (0.0%) com alvo definido


### Leitura da distribuição do alvo

A distribuição confirma um comportamento esperado da construção do alvo:

- **2024:** nenhum registro possui alvo definido, pois a base termina em 2024 e não existe `PEDE2025` para observar o resultado do ano seguinte;
- **2022 e 2023:** parte dos alunos possui alvo definido porque possui registro também no ano seguinte;
- alguns alunos não possuem alvo porque não há registro do mesmo `RA` no ano posterior.

A ausência de alvo em 2024, portanto, não representa um problema de qualidade ou erro de processamento. Ela é consequência direta da definição do problema preditivo e do período disponível na base.

As observações sem alvo não podem ser utilizadas para treinar ou avaliar o modelo de previsão do ano seguinte.

## 7. Padronização final: nomes de coluna e categorias

### Objetivo desta etapa

Depois da harmonização e das validações, aplicamos uma padronização final para deixar o painel consolidado consistente e adequado às etapas seguintes do projeto.

A função `padronizar_nomes_e_categorias()`, implementada em `src/data_prep.py`, recebe `painel_com_alvo` e realiza as seguintes alterações:

1. Renomeia as colunas para um padrão em letras maiúsculas.
2. Remove caracteres especiais dos nomes das colunas.

A padronização facilita o uso do mesmo conjunto de dados nos notebooks de análise exploratória e modelagem, reduzindo diferenças de nomenclatura entre as etapas do projeto.

In [ ]:
# Padroniza os nomes das colunas e as categorias do painel com alvo,
# preparando os dados para as próximas etapas e exibindo os cinco primeiros registros

painel_final = padronizar_nomes_e_categorias(painel_com_alvo)
painel_final.head(5)

,RA,ANO,FASE,FASE_IDEAL,DEFASAGEM_FORNECIDA,INDE,PEDRA,IAN,IDA,IEG,IAA,IPS,IPP,IPV,NOTA_MATEMATICA,NOTA_PORTUGUES,NOTA_INGLES,INDICADO,ATINGIU_PV,TURMA,GENERO,ANO_INGRESSO,INSTITUICAO_ENSINO,ANO_NASCIMENTO,DEFASAGEM_CALCULADA,IDADE_ANOS,ALVO_RISCO_DEFASAGEM_PROX_ANO
0,1,2022,7.0,8.0,-1,5.783,QUARTZO,5.0,4.0,4.1,8.3,5.6,NaN,7.278,2.7,3.5,6.0,SIM,NAO,A,FEMININO,2016,ESCOLA PUBLICA,2003,-1.0,19,0.0
1,1,2023,8.0,8.0,0,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8E,FEMININO,2016,PRIVADA PARCERIAS COM BOLSA 100,2003,0.0,20,0.0
2,1,2024,8.0,8.0,0,NaN,NaN,10.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8E,FEMININO,2021,PRIVADA PARCERIAS COM BOLSA 100,2003,0.0,21,NaN
3,2,2022,7.0,7.0,0,7.055,AMETISTA,10.0,6.8,5.2,8.8,6.3,NaN,6.778,6.3,4.5,9.7,NAO,NAO,A,FEMININO,2017,REDE DECISAO,2005,0.0,17,0.0
4,2,2023,8.0,8.0,0,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8E,FEMININO,2017,PRIVADA PARCERIAS COM BOLSA 100,2005,0.0,18,0.0


### Decisões deliberadas sobre o CSV final

Algumas decisões foram tomadas para preservar a informação original e evitar transformações que poderiam interferir nas análises posteriores:

- **Nenhuma coluna numérica é arredondada.** O `painel_final` mantém os valores numéricos com a precisão disponível no processamento. Não aplicamos arredondamento em `INDE`, `IDA`, notas ou outros indicadores.

- **O separador decimal permanece como ponto.** O CSV utiliza o padrão do `to_csv` do pandas, mantendo o ponto como separador decimal. Isso evita conversões adicionais quando o arquivo for utilizado posteriormente em ferramentas de análise e modelagem.

- **Valores ausentes permanecem ausentes.** Não substituímos valores ausentes por `0`, `"NULO"` ou strings vazias. A ausência é uma característica importante dos dados e, em vários casos, possui significado estrutural. Transformá-la artificialmente em um valor poderia fazer com que uma ausência fosse interpretada como uma observação real.

O tratamento específico dos valores ausentes será definido nas etapas em que cada variável for efetivamente utilizada.

## 8. Salvando o painel consolidado

### Objetivo desta etapa

Com as transformações e validações concluídas, salvamos o painel consolidado em formato CSV para que ele possa ser reutilizado nas próximas etapas do projeto.

O arquivo gerado é:

`data/processed/pede_painel_consolidado.csv`

Antes de finalizar esta etapa, também verificamos a quantidade de registros e as colunas presentes no arquivo salvo, garantindo que o resultado corresponde ao painel produzido durante o processamento.

In [ ]:
# Salva o painel final em CSV, confirma a criação e o tamanho do arquivo,
# relê o arquivo do disco e valida se sua estrutura e colunas permanecem
# iguais às do painel original em memória.

from pathlib import Path

caminho_saida = Path("../data/processed/pede_painel_consolidado.csv")
caminho_saida.parent.mkdir(parents=True, exist_ok=True)
painel_final.to_csv(caminho_saida, index=False)

print(f"Salvo em: {caminho_saida.resolve()}")
print(f"Existe no disco? {caminho_saida.exists()}")
print(f"Tamanho do arquivo: {caminho_saida.stat().st_size:,} bytes")

conferencia = pd.read_csv(caminho_saida)
print(f"\nRelido do disco: shape={conferencia.shape} (esperado: {painel_final.shape})")
assert conferencia.shape == painel_final.shape, "Shape do CSV relido não bate com o painel em memória"
assert list(conferencia.columns) == list(painel_final.columns), "Colunas do CSV relido não batem com painel_final"

Salvo em: /Users/sabrina/Documents/Cursos/FIAP/PósTech-DataAnalytics-2026/Fase5 /TCF5/tech-challenge-passos-magicos/data/processed/pede_painel_consolidado.csv
Existe no disco? True
Tamanho do arquivo: 407,879 bytes

Relido do disco: shape=(3030, 27) (esperado: (3030, 27))


## 9. Resumo

Neste notebook, transformamos as três bases anuais do PEDE em um único painel longitudinal, preservando a identificação anônima dos alunos e a granularidade de aluno-ano.

As principais etapas realizadas foram:

1. inspeção das diferenças estruturais entre `PEDE2022`, `PEDE2023` e `PEDE2024`;
2. harmonização dos nomes e formatos das variáveis;
3. extração e padronização das fases;
4. cálculo e validação da `defasagem_calculada`;
5. comparação da `Pedra` fornecida com a classificação recalculada a partir do INDE;
6. análise dos padrões de dados ausentes por ano;
7. avaliação da possibilidade de recomposição do `INDE` a partir do `IPP`;
8. criação do alvo de previsão de defasagem no ano seguinte;
9. padronização final dos nomes das colunas;
10. salvamento do painel consolidado.

### Principais decisões metodológicas

- diferenças estruturais entre anos foram preservadas e documentadas;
- a `defasagem_calculada` foi construída a partir de `fase_num - fase_ideal_num`;
- a `Pedra` fornecida pela fonte foi preservada, pois as faixas utilizadas no recálculo não reproduzem integralmente a classificação observada;
- valores ausentes não foram convertidos artificialmente em zero ou outros valores;
- o alvo preditivo foi definido com base no ano seguinte, evitando utilizar a situação do próprio ano como resposta;
- registros de 2024 não possuem alvo futuro porque não existe PEDE2025 na base disponível.

O resultado final desta etapa é o painel `data/processed/pede_painel_consolidado.csv`, que será utilizado como entrada para a análise exploratória e, posteriormente, para a modelagem preditiva.

In [ ]:
# Consolida e exibe os principais resultados de QA do painel, incluindo volume de dados,
# validação da defasagem e da Pedra, disponibilidade do IPP e do alvo por ano, além da
# confirmação da gravação e integridade do arquivo CSV final.

n_ipp_ausente_2022 = painel.loc[painel["ano"] == 2022, "ipp"].isna().mean()
n_ipp_preenchido_2023 = painel.loc[painel["ano"] == 2023, "ipp"].notna().mean()
prop_alvo_2022 = painel_com_alvo.loc[painel_com_alvo["ano"] == 2022, "alvo_risco_defasagem_prox_ano"].notna().mean()
prop_alvo_2023 = painel_com_alvo.loc[painel_com_alvo["ano"] == 2023, "alvo_risco_defasagem_prox_ano"].notna().mean()

linhas_resumo = [
    f"- Painel consolidado: {painel_com_alvo.shape[0]} linhas (aluno-ano) x {painel_com_alvo.shape[1]} colunas, "
    f"cobrindo {painel_com_alvo['ra'].nunique()} alunos (RA) distintos nos anos "
    f"{sorted(painel_com_alvo['ano'].unique().tolist())}.",
    f"- QA defasagem: {taxa_match:.2%} de concordância entre defasagem_calculada e defasagem_fornecida "
    f"(correlação {correlacao:.4f}) em {len(comparavel)} linhas comparáveis -> extração de fase validada.",
    f"- QA Pedra: {taxa_match_pedra:.2%} de concordância entre a Pedra recalculada a partir do INDE e a Pedra "
    f"fornecida, em {len(comparavel_pedra)} linhas comparáveis -> divergência concentrada em fronteiras de "
    f"faixa, não em erro de parsing (ver crosstab da seção 4).",
    f"- ipp: {n_ipp_ausente_2022:.0%} ausente em 2022 (indicador não existia) vs. "
    f"{n_ipp_preenchido_2023:.0%} preenchido em 2023.",
    f"- Alvo (alvo_risco_defasagem_prox_ano): definido para {prop_alvo_2022:.1%} dos alunos de 2022 e "
    f"{prop_alvo_2023:.1%} dos de 2023; indefinido (NA) para 100% de 2024, por não existir PEDE2025 na base.",
    f"- Arquivo salvo em data/processed/pede_painel_consolidado.csv ({caminho_saida.stat().st_size:,} bytes), "
    f"relido do disco e conferido contra o DataFrame em memória.",
]
print("\n".join(linhas_resumo))

- Painel consolidado: 3030 linhas (aluno-ano) x 27 colunas, cobrindo 1661 alunos (RA) distintos nos anos [2022, 2023, 2024].
- QA defasagem: 99.93% de concordância entre defasagem_calculada e defasagem_fornecida (correlação 0.9960) em 3030 linhas comparáveis -> extração de fase validada.
- QA Pedra: 81.51% de concordância entre a Pedra recalculada a partir do INDE e a Pedra fornecida, em 2845 linhas comparáveis -> divergência concentrada em fronteiras de faixa, não em erro de parsing (ver crosstab da seção 4).
- ipp: 100% ausente em 2022 (indicador não existia) vs. 93% preenchido em 2023.
- Alvo (alvo_risco_defasagem_prox_ano): definido para 69.8% dos alunos de 2022 e 75.4% dos de 2023; indefinido (NA) para 100% de 2024, por não existir PEDE2025 na base.
- Arquivo salvo em data/processed/pede_painel_consolidado.csv (407,879 bytes), relido do disco e conferido contra o DataFrame em memória.
